# First Layer Weight Visualization

This notebook visualizes the first layer weights of the trained MLP model.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.model import ThreeLayerNN
from src.data_loader import load_eurosat_data

In [ ]:
# Load the trained model
model_path = '../output/experiment_20260412_123429/checkpoints/best_model.pkl'

with open(model_path, 'rb') as f:
    checkpoint = pickle.load(f)

print("Checkpoint keys:", checkpoint.keys())
print("\nModel parameters:")
for key, value in checkpoint['params'].items():
    print(f"  {key}: shape {value.shape}")

In [ ]:
# Extract first layer weights
# W1 shape: (12288, 256) - input_dim to hidden_dim
W1 = checkpoint['params']['W1']
print(f"W1 shape: {W1.shape}")
print(f"W1 range: [{W1.min():.4f}, {W1.max():.4f}]")

In [ ]:
def visualize_neuron_weight(W, neuron_idx, ax, title=None):
    """
    Visualize the weight of a single neuron in the first hidden layer.
    
    W: weight matrix (input_dim, hidden_dim)
    neuron_idx: which neuron to visualize
    """
    # Get weights for this neuron (connections from all input pixels)
    w = W[:, neuron_idx]  # shape: (12288,)
    
    # Reshape to 64x64x3 image
    w_img = w.reshape(64, 64, 3)
    
    # Normalize to [0, 1] for visualization
    w_min, w_max = w_img.min(), w_img.max()
    if w_max - w_min > 1e-8:
        w_img = (w_img - w_min) / (w_max - w_min)
    
    ax.imshow(w_img)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=10)
    
    return w_img

In [ ]:
# Visualize 16 randomly selected neurons
np.random.seed(42)
hidden_dim = W1.shape[1]
selected_neurons = np.random.choice(hidden_dim, 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('First Layer Weights (16 Random Neurons)', fontsize=14)

for idx, neuron_idx in enumerate(selected_neurons):
    row, col = idx // 4, idx % 4
    visualize_neuron_weight(W1, neuron_idx, axes[row, col], 
                           title=f'Neuron {neuron_idx}')

plt.tight_layout()
plt.savefig('first_layer_weights_random.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze weight statistics for each neuron
# Find neurons with different characteristics

# Compute variance for each neuron's weights
neuron_variances = np.var(W1, axis=0)
neuron_means = np.mean(W1, axis=0)

# Find neurons with highest/lowest variance
high_var_neurons = np.argsort(neuron_variances)[-4:][::-1]
low_var_neurons = np.argsort(neuron_variances)[:4]

# Find neurons with positive/negative mean
positive_mean_neurons = np.argsort(neuron_means)[-4:][::-1]
negative_mean_neurons = np.argsort(neuron_means)[:4]

print("High variance neurons:", high_var_neurons)
print("Low variance neurons:", low_var_neurons)

In [ ]:
# Visualize high variance neurons (most distinctive patterns)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle('Neurons with Highest Weight Variance', fontsize=12)

for idx, neuron_idx in enumerate(high_var_neurons):
    visualize_neuron_weight(W1, neuron_idx, axes[idx], 
                           title=f'Neuron {neuron_idx}\nVar: {neuron_variances[neuron_idx]:.6f}')

plt.tight_layout()
plt.savefig('first_layer_weights_high_var.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize low variance neurons
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle('Neurons with Lowest Weight Variance', fontsize=12)

for idx, neuron_idx in enumerate(low_var_neurons):
    visualize_neuron_weight(W1, neuron_idx, axes[idx], 
                           title=f'Neuron {neuron_idx}\nVar: {neuron_variances[neuron_idx]:.6f}')

plt.tight_layout()
plt.savefig('first_layer_weights_low_var.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualize Weights by Output Class

Since this is a 3-layer MLP, we can analyze how the first layer weights contribute to each output class through the network.

In [ ]:
# Load data to get class names
data = load_eurosat_data('../EuroSAT_RGB', batch_size=64, val_ratio=0.15, test_ratio=0.15)
class_names = data['class_names']
print("Classes:", class_names)

In [ ]:
# Get all layer weights
W1 = checkpoint['params']['W1']  # (12288, 256)
W2 = checkpoint['params']['W2']  # (256, 256)
W3 = checkpoint['params']['W3']  # (256, 10)

print(f"W1: {W1.shape}")
print(f"W2: {W2.shape}")
print(f"W3: {W3.shape}")

In [ ]:
# Compute contribution of input pixels to each output class
# This is a simplified view: W1 @ W2 @ W3 gives approximate pixel-to-class weights
# Shape: (12288, 10)

pixel_to_class = W1 @ W2 @ W3  # (12288, 10)
print(f"Pixel-to-class weight shape: {pixel_to_class.shape}")

In [ ]:
# Visualize 4 example classes
selected_classes = [0, 2, 4, 7]  # AnnualCrop, HerbaceousVegetation, Industrial, Residential

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, class_idx in enumerate(selected_classes):
    # Get weights for this class
    w = pixel_to_class[:, class_idx]
    w_img = w.reshape(64, 64, 3)
    
    # Normalize
    w_min, w_max = w_img.min(), w_img.max()
    if w_max - w_min > 1e-8:
        w_img_norm = (w_img - w_min) / (w_max - w_min)
    else:
        w_img_norm = w_img
    
    # Original normalized
    axes[0, idx].imshow(w_img_norm)
    axes[0, idx].set_title(f'{class_names[class_idx]}\n(Normalized)', fontsize=10)
    axes[0, idx].axis('off')
    
    # With positive/negative highlighting
    # Red = positive weights, Blue = negative weights
    w_centered = w_img - w_img.mean()
    w_pos = np.maximum(w_centered, 0)
    w_neg = np.maximum(-w_centered, 0)
    
    # Normalize for visualization
    if w_pos.max() > 0:
        w_pos = w_pos / w_pos.max()
    if w_neg.max() > 0:
        w_neg = w_neg / w_neg.max()
    
    combined = np.zeros((64, 64, 3))
    combined[:, :, 0] = w_pos.mean(axis=2)  # Red for positive
    combined[:, :, 2] = w_neg.mean(axis=2)  # Blue for negative
    
    axes[1, idx].imshow(combined)
    axes[1, idx].set_title(f'{class_names[class_idx]}\n(R=Pos, B=Neg)', fontsize=10)
    axes[1, idx].axis('off')

fig.suptitle('Pixel-to-Class Weight Visualization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('pixel_to_class_weights.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Another approach: Visualize which neurons most strongly activate each class
# For each output class, find the hidden neurons with highest W3 weights

fig, axes = plt.subplots(10, 4, figsize=(14, 28))

for class_idx in range(10):
    # Get W3 weights for this class
    w3_class = W3[:, class_idx]  # (256,)
    
    # Find top 4 neurons
    top_neurons = np.argsort(np.abs(w3_class))[-4:][::-1]
    
    for idx, neuron_idx in enumerate(top_neurons):
        visualize_neuron_weight(W1, neuron_idx, axes[class_idx, idx],
                               title=f'N{neuron_idx}, W3={w3_class[neuron_idx]:.3f}')
    
    # Add class label on the left
    axes[class_idx, 0].set_ylabel(class_names[class_idx], fontsize=11, rotation=0, 
                                   labelpad=80, ha='right', va='center')

fig.suptitle('Top 4 Neurons for Each Output Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('neurons_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select 4 classes for report
selected_classes = [0, 3, 5, 9]  # AnnualCrop, Highway, Pasture, SeaLake

fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for row_idx, class_idx in enumerate(selected_classes):
    w3_class = W3[:, class_idx]
    top_neurons = np.argsort(np.abs(w3_class))[-4:][::-1]
    
    for col_idx, neuron_idx in enumerate(top_neurons):
        visualize_neuron_weight(W1, neuron_idx, axes[row_idx, col_idx],
                               title=f'N{neuron_idx}')
    
    axes[row_idx, 0].set_ylabel(class_names[class_idx], fontsize=11, rotation=0,
                                 labelpad=70, ha='right', va='center')

fig.suptitle('Top 4 Neurons for Selected Classes', fontsize=14)
plt.tight_layout()
plt.savefig('neurons_by_class_selected.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
# Summary statistics
print("=== Weight Statistics ===")
print(f"\nW1 (Input -> Hidden1):")
print(f"  Shape: {W1.shape}")
print(f"  Mean: {W1.mean():.6f}")
print(f"  Std: {W1.std():.6f}")
print(f"  Range: [{W1.min():.4f}, {W1.max():.4f}]")

print(f"\nW2 (Hidden1 -> Hidden2):")
print(f"  Shape: {W2.shape}")
print(f"  Mean: {W2.mean():.6f}")
print(f"  Std: {W2.std():.6f}")

print(f"\nW3 (Hidden2 -> Output):")
print(f"  Shape: {W3.shape}")
print(f"  Mean: {W3.mean():.6f}")
print(f"  Std: {W3.std():.6f}")